In [7]:
import warnings
import os
warnings.filterwarnings('ignore')
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.templates.default = "plotly_white"
pio.kaleido.scope.mathjax = None
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from itertools import cycle
from rasterstats import zonal_stats
%load_ext autoreload
%autoreload 2
np.random.seed()
tqdm.pandas()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# Assumes this is a point file in a geographic coordinate system like WGS84 (EPSG:4326)
stations = gpd.read_file("stations.geojson")

In [9]:
# For Accra, Ghana, UTM Zone 30N (EPSG:32630) uses meters as units.
stations_metric = stations.to_crs(epsg=32630)

In [10]:
# 3. Create your localized LUR impact buffer rings (e.g., 500m and 1000m)
print("Generating spatial buffers around ground stations...")
buffers_500m = stations_metric.copy()
buffers_500m['geometry'] = stations_metric.geometry.buffer(500)

buffers_1000m = stations_metric.copy()
buffers_1000m['geometry'] = stations_metric.geometry.buffer(1000)

# 4. Project buffers back to standard WGS84 to match most raster alignments
buffers_500m_wgs84 = buffers_500m.to_crs(epsg=4326)
buffers_1000m_wgs84 = buffers_1000m.to_crs(epsg=4326)

Generating spatial buffers around ground stations...


In [11]:
# 5. Define the path to your raw un-converted population density raster
raster_path = "GHA15adj_040213.tif"

# 6. Extract Zonal Statistics directly from the raster
print("Extracting population statistics inside buffers (this takes seconds)...")

# Calculate the mean pixel value (density) inside the 500m circles
stats_500m = zonal_stats(
    buffers_500m_wgs84, 
    raster_path, 
    stats="mean", 
    geojson_out=False
)
# Extract the list of dictionaries into a clean column
stations['pop_density_500m'] = [x['mean'] for x in stats_500m]

# Calculate the mean pixel value (density) inside the 1000m circles
stats_1000m = zonal_stats(
    buffers_1000m_wgs84, 
    raster_path, 
    stats="mean", 
    geojson_out=False
)
stations['pop_density_1000m'] = [x['mean'] for x in stats_1000m]

Extracting population statistics inside buffers (this takes seconds)...


In [12]:
stations.head()

,sensor_name,datetime,value,latitude,longitude,geometry,pop_density_500m,pop_density_1000m
0,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (-0.18587 5.65384),7.2617,15.498734
1,Afri-SET,2024-01-02,NaN,5.653842,-0.185875,POINT (-0.18587 5.65384),7.2617,15.498734
2,Afri-SET,2024-01-03,NaN,5.653842,-0.185875,POINT (-0.18587 5.65384),7.2617,15.498734
3,Afri-SET,2024-01-04,NaN,5.653842,-0.185875,POINT (-0.18587 5.65384),7.2617,15.498734
4,Afri-SET,2024-01-05,NaN,5.653842,-0.185875,POINT (-0.18587 5.65384),7.2617,15.498734


In [13]:
metric_crs = "EPSG:32630"

In [14]:
unpaved_roads  = gpd.read_file("unpave.geojson").to_crs(metric_crs)
paved_roads    = gpd.read_file("roads.geojson").to_crs(metric_crs)
settlements    = gpd.read_file("settlement.geojson").to_crs(metric_crs)
industrial     = gpd.read_file("export.geojson").to_crs(metric_crs)
car = gpd.read_file('car_stations.geojson').to_crs(metric_crs)

In [19]:
car.head()

,id,amenity,industrial,surface,landuse,geometry
0,way/17914757,bus_station,NaN,NaN,NaN,"POLYGON ((807441.295 615619.073, 807597.679 61..."
1,way/238521540,bus_station,NaN,NaN,commercial,"POLYGON ((812326.931 618533.570, 812371.245 61..."
2,way/295785262,NaN,NaN,NaN,landfill,"POLYGON ((807101.637 614728.968, 807063.318 61..."
3,way/298830279,bus_station,NaN,NaN,NaN,"POLYGON ((807992.050 616203.596, 808006.757 61..."
4,way/300519834,bus_station,NaN,NaN,NaN,"POLYGON ((813224.914 622008.812, 813150.922 62..."


In [22]:
# 1. Isolate Landfills and ensure they are strictly Polygons/MultiPolygons
raw_landfills = car[
    (car['landuse'] == 'landfill') | (car['amenity'] == 'landfill')
]
landfills = raw_landfills[raw_landfills.geometry.geom_type.str.contains('Polygon', na=False)].copy()

In [23]:
# 2. Isolate Bus Stations and ensure they are strictly Polygons/MultiPolygons
raw_bus_stations = car[car['amenity'] == 'bus_station']
bus_stations = raw_bus_stations[raw_bus_stations.geometry.geom_type.str.contains('Polygon', na=False)].copy()

print(f"📋 Polygons Isolated: Found {len(landfills)} landfills and {len(bus_stations)} bus station polygons.")

📋 Polygons Isolated: Found 3 landfills and 17 bus station polygons.


In [15]:
stations = stations.to_crs(epsg=32630)

In [16]:
# 1. Generate unique integer codes for each unique station name
station_codes, station_names = pd.factorize(stations['sensor_name'])

# 2. Add it to your dataframe as a formatted string column (e.g., STN_0, STN_1)
stations['station_id'] = [f"STN_{code}" for code in station_codes]

In [17]:
stations.head()

,sensor_name,datetime,value,latitude,longitude,geometry,pop_density_500m,pop_density_1000m,station_id
0,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0
1,Afri-SET,2024-01-02,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0
2,Afri-SET,2024-01-03,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0
3,Afri-SET,2024-01-04,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0
4,Afri-SET,2024-01-05,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0


In [18]:
# Create a clean baseline dataframe to hold the extracted predictors
df_features = pd.DataFrame(stations[['station_id']])

In [24]:
buffer_distances = [300, 1000]

for distance in buffer_distances:
    print(f"\nProcessing LUR spatial predictors inside {distance}m buffers...")
    
    # Generate temporary circular geometries for the target distance
    stations_buffered = stations.copy()
    stations_buffered['geometry'] = stations.geometry.buffer(distance)
    
    # --- A. LINE FEATURES: ROAD LENGTHS ---
    # Unpaved roads length
    unpaved_clip = gpd.overlay(unpaved_roads, stations_buffered, how='intersection')
    unpaved_clip['length_m'] = unpaved_clip.geometry.length
    unpaved_lengths = unpaved_clip.groupby('station_id')['length_m'].sum()
    df_features[f'unpaved_road_length_{distance}m'] = df_features['station_id'].map(unpaved_lengths).fillna(0)
    
    # Paved roads length
    paved_clip = gpd.overlay(paved_roads, stations_buffered, how='intersection')
    paved_clip['length_m'] = paved_clip.geometry.length
    paved_lengths = paved_clip.groupby('station_id')['length_m'].sum()
    df_features[f'paved_road_length_{distance}m'] = df_features['station_id'].map(paved_lengths).fillna(0)
    
    
    # --- B. POLYGON FEATURES: LAND USE FOOTPRINTS ---
    total_buffer_area = stations_buffered.geometry.area.iloc[0] 
    
    # Settlements footprint area & percentage index
    settlement_clip = gpd.overlay(settlements, stations_buffered, how='intersection')
    settlement_clip['area_m2'] = settlement_clip.geometry.area
    settlement_areas = settlement_clip.groupby('station_id')['area_m2'].sum()
    df_features[f'settlement_area_{distance}m'] = df_features['station_id'].map(settlement_areas).fillna(0)
    df_features[f'settlement_pct_{distance}m']  = (df_features[f'settlement_area_{distance}m'] / total_buffer_area) * 100
    
    # Industrial areas footprint
    industrial_clip = gpd.overlay(industrial, stations_buffered, how='intersection')
    industrial_clip['area_m2'] = industrial_clip.geometry.area
    industrial_areas = industrial_clip.groupby('station_id')['area_m2'].sum()
    df_features[f'industrial_area_{distance}m'] = df_features['station_id'].map(industrial_areas).fillna(0)

    # 🆕 Landfill areas footprint inside the buffer
    if not landfills.empty:
        landfill_clip = gpd.overlay(landfills, stations_buffered, how='intersection')
        landfill_clip['area_m2'] = landfill_clip.geometry.area
        landfill_areas = landfill_clip.groupby('station_id')['area_m2'].sum()
        df_features[f'landfill_area_{distance}m'] = df_features['station_id'].map(landfill_areas).fillna(0)
    else:
        df_features[f'landfill_area_{distance}m'] = 0.0

    # 🆕 Bus Station areas footprint inside the buffer
    if not bus_stations.empty:
        bus_station_clip = gpd.overlay(bus_stations, stations_buffered, how='intersection')
        bus_station_clip['area_m2'] = bus_station_clip.geometry.area
        bus_station_areas = bus_station_clip.groupby('station_id')['area_m2'].sum()
        df_features[f'bus_station_area_{distance}m'] = df_features['station_id'].map(bus_station_areas).fillna(0)
    else:
        df_features[f'bus_station_area_{distance}m'] = 0.0

print("\n✅ All LUR spatial predictors successfully extracted!")


Processing LUR spatial predictors inside 300m buffers...

Processing LUR spatial predictors inside 1000m buffers...

✅ All LUR spatial predictors successfully extracted!


In [25]:
# 1. Create a copy of the original geometric points layer
stations_output = stations.copy()

# 2. Merge all the newly calculated buffer features back onto it using station_id
# This brings along your unpaved_road_lengths, settlement_pcts, etc.
stations_output = stations_output.merge(df_features, on='station_id')

In [29]:
stations_output.head()

,sensor_name,datetime,value,latitude,longitude,geometry,pop_density_500m,pop_density_1000m,station_id,unpaved_road_length_300m,...,industrial_area_300m,landfill_area_300m,bus_station_area_300m,unpaved_road_length_1000m,paved_road_length_1000m,settlement_area_1000m,settlement_pct_1000m,industrial_area_1000m,landfill_area_1000m,bus_station_area_1000m
0,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0,71480.840116,...,0.0,0.0,0.0,3.930313e+06,2.396685e+06,4.294981e+06,136.933358,0.0,0.0,3.982433e+06
1,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0,71480.840116,...,0.0,0.0,0.0,3.930313e+06,2.396685e+06,4.294981e+06,136.933358,0.0,0.0,3.982433e+06
2,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0,71480.840116,...,0.0,0.0,0.0,3.930313e+06,2.396685e+06,4.294981e+06,136.933358,0.0,0.0,3.982433e+06
3,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0,71480.840116,...,0.0,0.0,0.0,3.930313e+06,2.396685e+06,4.294981e+06,136.933358,0.0,0.0,3.982433e+06
4,Afri-SET,2024-01-01,NaN,5.653842,-0.185875,POINT (811752.235 625694.493),7.2617,15.498734,STN_0,71480.840116,...,0.0,0.0,0.0,3.930313e+06,2.396685e+06,4.294981e+06,136.933358,0.0,0.0,3.982433e+06


In [31]:
output_csv_path = "station_urban_predictors.csv"
stations_output.to_csv(output_csv_path, index=False)

print("\n🎉 All local urban buffer features compiled successfully!")
print(f"File saved to: {output_csv_path}")


🎉 All local urban buffer features compiled successfully!
File saved to: station_urban_predictors.csv


In [27]:
df_features.columns

Index(['station_id', 'unpaved_road_length_300m', 'paved_road_length_300m',
       'settlement_area_300m', 'settlement_pct_300m', 'industrial_area_300m',
       'landfill_area_300m', 'bus_station_area_300m',
       'unpaved_road_length_1000m', 'paved_road_length_1000m',
       'settlement_area_1000m', 'settlement_pct_1000m',
       'industrial_area_1000m', 'landfill_area_1000m',
       'bus_station_area_1000m'],
      dtype='object')

In [ ]:
# 1. Create a copy of the original geometric points layer
stations_output = stations.copy()

# 2. Merge all the newly calculated buffer features back onto it using station_id
# This brings along your unpaved_road_lengths, settlement_pcts, etc.
stations_output = stations_output.merge(df_features, on='station_id')

# 3. CRITICAL: GeoJSON files must be saved in standard WGS84 coordinates (degrees)
# if you plan on opening them in web maps or sharing them. 
stations_output = stations_output.to_crs(epsg=4326)

# 4. Save the file! .to_file() will now work perfectly
output_spatial_path = "station_urban_predictors.geojson"
stations_output.to_file(output_spatial_path, driver="GeoJSON")

print(f"\n🎉 Success! Spatial GeoJSON safely compiled and saved to: {output_spatial_path}")

In [ ]:
new.head()